# Prepare the real SDSS-RM cache (instructor only)
Grier et al. (2017), ApJ 851, 21. CDS catalogue J/ApJ/851/21. [ReadMe](https://cdsarc.cds.unistra.fr/viz-bin/ReadMe/J/ApJ/851/21?format=html).

Run all cells in Google Colab. This downloads original catalogue tables, extracts RMIDs 101, 16, 33, validates the data against the practical's loader and packages the two CSVs. No simulated measurements or lag solutions are included. If automatic retrieval fails, upload the two original fixed-width tables (.dat or .dat.gz) to the specified cache directory and rerun. **Do not distribute a ZIP unless all checks pass.**

In [ ]:
%pip install -q astroquery

In [ ]:
# If astroquery is missing in Colab, run: %pip install -q astroquery
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy
from scipy.linalg import cho_factor, cho_solve
from IPython.display import display

IDS = (101, 16, 33)
SEED = 2026
MIN_PAIRS = 10
LAGS = np.arange(-100., 101., 1.)
ALT_LAGS = np.arange(-70., 71., 1.)
CENTROID_FRACTION = 0.8
MIN_R = 0.5   # teaching screen, NOT a significance threshold
B = 500       # final run; 100 for a quick trial, then restore 500
CACHE = Path('data_cache/sdss_rm_2017')
CACHE.mkdir(parents=True, exist_ok=True)
print(sys.version.split()[0], np.__version__, pd.__version__, scipy.__version__)

In [ ]:
import requests, gzip, io
BASES = ['https://cdsarc.cds.unistra.fr/ftp/J/ApJ/851/21/',
         'https://cdsarc.u-strasbg.fr/ftp/J/ApJ/851/21/']
for stem, expected, width in [('table1',222,72),('table2',152258,54)]:
    if any((CACHE / (stem+s)).is_file() for s in ('.dat','.dat.gz')):
        continue
    done = False
    for base in BASES:
        for suffix in ('.dat.gz','.dat'):
            try:
                response = requests.get(base+stem+suffix, timeout=90)
                response.raise_for_status()
                data = response.content
                if data[:2] == b'\x1f\x8b':
                    data = gzip.decompress(data)
                lines = data.decode('ascii').splitlines()
                assert len(lines)==expected and all(len(x)>=width and x[:3].strip().isdigit() for x in lines), 'Unexpected table format or row count'
                (CACHE/(stem+'.dat')).write_bytes(data)
                print('Retrieved', stem, len(lines), 'rows from', response.url)
                done = True
                break
            except Exception as exc:
                print('Download failed:', base+stem+suffix, str(exc))
        if done: break
    if not done: print('Will try VizieR query for',stem)


In [ ]:
# Supplied loader: use local CDS fixed-width files, or query VizieR by RMID.
# Manual fallback: download table1.dat and table2.dat (or .gz) from
# https://cdsarc.cds.unistra.fr/ftp/J/ApJ/851/21/ into CACHE.
# Fixed-width positions come directly from the CDS ReadMe.
T1_COLS = ['RMID', 'SDSS', 'RAdeg', 'DEdeg', 'z', 'imag', 'SNR',
           'logAGN', 'logHost', 'logM']
T1_SPEC = [(0,3),(4,23),(24,32),(33,40),(41,46),(47,53),
           (54,58),(59,63),(64,68),(69,72)]
T2_COLS = ['RMID','MJD','Band','Tel','Flux','e_Flux','S','e_S','Flag']
T2_SPEC = [(0,3),(4,13),(14,20),(21,22),(23,30),(31,37),
           (38,45),(46,52),(53,54)]

def local_table(stem, cols, spec):
    for suffix in ('.dat', '.dat.gz'):
        path = CACHE / (stem + suffix)
        if path.is_file():
            print('Reading', path)
            return pd.read_fwf(path, colspecs=spec, names=cols)
    return None

def query_one(table, cols, rmid):
    from astroquery.vizier import Vizier
    v = Vizier(columns=cols, row_limit=-1, timeout=120)
    result = v.query_constraints(catalog='J/ApJ/851/21/' + table,
                                 RMID=str(rmid))
    if len(result) != 1:
        raise RuntimeError(f'Unexpected VizieR result for {table}, RMID {rmid}')
    return result[0].to_pandas()

def fetch_tables():
    meta = local_table('table1', T1_COLS, T1_SPEC)
    obs = local_table('table2', T2_COLS, T2_SPEC)
    # Cache only the assigned real observations for offline re-runs.
    paths = [CACHE / 'assigned_meta.csv', CACHE / 'assigned_lc.csv']
    if meta is None:
        if paths[0].exists():
            print('Reading cached metadata:', paths[0])
            meta = pd.read_csv(paths[0])
        else:
            meta = pd.concat([query_one('table1', ['RMID','z'], i)
                              for i in IDS], ignore_index=True)
    if obs is None:
        if paths[1].exists():
            print('Reading cached observations:', paths[1])
            obs = pd.read_csv(paths[1])
        else:
            obs = pd.concat([query_one('table2', T2_COLS, i)
                             for i in IDS], ignore_index=True)
    for df in (meta, obs):
        df['RMID'] = pd.to_numeric(df['RMID'], errors='raise').astype(int)
    meta = meta.loc[meta.RMID.isin(IDS)].copy()
    obs = obs.loc[obs.RMID.isin(IDS)].copy()
    assert set(meta.RMID) == set(IDS) and set(obs.RMID) == set(IDS)
    assert not meta.RMID.duplicated().any()
    assert set(T2_COLS).issubset(obs.columns)
    for col in ['Band', 'Tel']:
        obs[col] = obs[col].map(lambda x: x.decode() if isinstance(x, bytes) else str(x)).str.strip()
    meta.to_csv(paths[0], index=False)
    obs.to_csv(paths[1], index=False)
    return meta, obs

meta, obs = fetch_tables()
for col in ['MJD','Flux','e_Flux','S','e_S','Flag']:
    obs[col] = pd.to_numeric(obs[col], errors='coerce')
meta['z'] = pd.to_numeric(meta['z'], errors='raise')
assert np.isfinite(meta.z).all() and (meta.z > 0).all()
assert obs.MJD.between(6649,6857).all()
assert obs.Flag.isin([0,1]).all()
assert obs.Band.isin(['g','i','Halpha','Hbeta']).all()
assert obs.Tel.isin(['C','B','S']).all()
display(meta[['RMID','z']])
display(obs.groupby(['RMID','Band','Tel']).size().rename('raw_rows').reset_index())

In [ ]:
def clean_band(raw, band, flux, err):
    a = raw.loc[(raw.Band == band) & (raw.Flag == 0)].copy()
    ok = (np.isfinite(a.MJD) & np.isfinite(a[flux]) &
          np.isfinite(a[err]) & (a[err] > 0))
    return a.loc[ok].copy()

def combine_epochs(a, flux, err, t0):
    b = pd.DataFrame({'t': a.MJD.to_numpy(float) - t0,
                      'f': a[flux].to_numpy(float),
                      'e': a[err].to_numpy(float)})
    b['w'] = 1 / b.e**2
    b['wf'] = b.w * b.f
    s = b.groupby('t', sort=True)[['w','wf']].sum()
    return np.column_stack([s.index.to_numpy(float), s.wf/s.w,
                            1/np.sqrt(s.w)])

def make_pair(rmid, band='g'):
    raw = obs.loc[obs.RMID == rmid]
    cont = clean_band(raw, band, 'Flux', 'e_Flux')
    line = clean_band(raw, 'Hbeta', 'S', 'e_S')
    line = line.loc[line.Tel == 'S']
    counts = cont.groupby('Tel').MJD.nunique()
    if counts.empty:
        raise ValueError(f'No valid {band} continuum for RMID {rmid}')
    tel = sorted(counts.index[counts == counts.max()])[0]
    cont = cont.loc[cont.Tel == tel]
    if min(cont.MJD.nunique(), line.MJD.nunique()) < MIN_PAIRS:
        raise ValueError(f'Insufficient valid epochs for RMID {rmid}')
    t0 = min(cont.MJD.min(), line.MJD.min())
    return {'c': combine_epochs(cont, 'Flux', 'e_Flux', t0),
            'l': combine_epochs(line, 'S', 'e_S', t0),
            'z': float(meta.loc[meta.RMID == rmid,'z'].iloc[0]),
            'tel': tel, 'band': band, 't0': t0,
            'raw_c': len(raw.loc[(raw.Band == band)&(raw.Tel == tel)]),
            'raw_l': len(raw.loc[(raw.Band == 'Hbeta')&(raw.Tel == 'S')]),
            'valid_c': len(cont), 'valid_l': len(line)}

pairs = {i: make_pair(i) for i in IDS}

In [ ]:
# Package only after successful validation and construction of all light-curve pairs.
from zipfile import ZipFile, ZIP_DEFLATED
import hashlib
for i,p in pairs.items():
    print(i, 'continuum epochs:', len(p['c']), 'Hbeta epochs:', len(p['l']))
zip_path = Path('SDSS_RM_2017_data_cache.zip')
with ZipFile(zip_path, 'w', ZIP_DEFLATED) as zf:
    for name in ('assigned_meta.csv','assigned_lc.csv'):
        path = CACHE/name
        zf.write(path, arcname=str(path))
        print(name, 'SHA256:', hashlib.sha256(path.read_bytes()).hexdigest())
    zf.writestr('CACHE_README.txt', 'Source: Grier et al. (2017), ApJ 851, 21; CDS J/ApJ/851/21; DOI 10.26093/cds/vizier.18510021. RMIDs 101,16,33. All available bands and flags retained. MJD is MJD-50000. Extract beside the Week 7 notebook.\n')
print('Created',zip_path)
try:
    from google.colab import files
    files.download(str(zip_path))
except ImportError:
    pass